# P1-25 · Second reader: marked images -> masks -> inter-reader agreement and model scores (CPU, ~5 min)
**Runtime: CPU.** No training. Read-only for QSEG_FINAL, reader1 annotations and all run folders. Writes only `MyDrive/P1_v2/annot_results/reader2/`.

Input: the 20 images marked by the second reader on a phone (red line = breast boundary, yellow line = pectoral boundary), uploaded as they are
(any file names) to `MyDrive/P1_v2/annot_test/reader2_marked/`.
1. Each marked image is matched to its original by image content (not by file name).
2. Red and yellow strokes are extracted by color; the breast region is everything on the chest-wall side of the red line,
   the pectoral region is the part of the breast region cut off by the yellow line at the upper chest-wall corner.
3. Masks are mapped to the 256 x 256 model input exactly as reader1 (same padding/resizing) and compared with reader1, the rule-based proxy
   and every saved model run. QC sheets show both readers' outlines for visual checking.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, glob, json, numpy as np, pandas as pd, cv2
ROOT = '/content/drive/MyDrive/QSEG_FINAL'; OUT = '/content/drive/MyDrive/P1_v2'
MARKED = f'{OUT}/annot_test/reader2_marked'; RES = f'{OUT}/annot_results/reader2'; os.makedirs(RES, exist_ok=True)
import sys; sys.path.insert(0, ROOT + '/code')
from qseg.data_prep import generate_breast_mask, pad_to_square, chest_wall_side, canonical_id
# the 20 test images given to the second reader; the list holds coded patient identifiers, so it is kept privately in Drive
SECOND = pd.read_csv(f'{OUT}/annot_test/reader2_selection.csv').original_filename.tolist()
man = pd.read_csv(ROOT + '/results/data/manifest.csv'); test = man[man.split == 'test'].reset_index(drop=True)
sub = test[test.original_filename.isin(SECOND)].reset_index(drop=True)
print('second-reader images in the test manifest:', len(sub), 'of', len(SECOND)); assert len(sub) == 20
raw = {}
for f in sub.original_filename:
    hits = [h for h in glob.glob(f'{ROOT}/**/{glob.escape(f)}', recursive=True) if '/results/' not in h]
    raw[f] = hits[0] if hits else None
assert all(raw.values()), [k for k, v in raw.items() if v is None]
files = sorted(f for f in glob.glob(f'{MARKED}/*') if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')))
print('marked files:', len(files)); assert len(files) == 20, 'expected 20 marked images in ' + MARKED

In [ ]:
# 1 · Match every marked image to its original by content (strokes removed), check one-to-one and aspect ratio
def color_masks(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV); h, s, v = hsv[..., 0].astype(int), hsv[..., 1].astype(int), hsv[..., 2].astype(int)
    red = ((h <= 10) | (h >= 165)) & (s >= 90) & (v >= 70)
    yel = (h >= 18) & (h <= 40) & (s >= 90) & (v >= 90)
    return red, yel
def thumb(gray, mask=None, n=96):
    g = gray.astype(np.float32)
    if mask is not None and mask.any(): g[mask] = cv2.medianBlur(gray, 15)[mask]
    t = cv2.resize(g, (n, n), interpolation=cv2.INTER_AREA); t -= t.mean(); return t / (t.std() + 1e-6)
ORIG = {f: cv2.imread(raw[f], cv2.IMREAD_GRAYSCALE) for f in sub.original_filename}
OT = {f: thumb(g) for f, g in ORIG.items()}
rows, MK = [], {}
for p in files:
    bgr = cv2.imread(p); red, yel = color_masks(bgr); g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    t = thumb(g, red | yel); sc = {f: float((t * o).mean()) for f, o in OT.items()}; best = sorted(sc, key=sc.get, reverse=True)
    H, W = ORIG[best[0]].shape
    rows.append(dict(marked=os.path.basename(p), original=best[0], match_corr=sc[best[0]], second_best=sc[best[1]],
                     aspect_marked=bgr.shape[1] / bgr.shape[0], aspect_orig=W / H, red_px=int(red.sum()), yellow_px=int(yel.sum())))
    MK[best[0]] = (bgr, red, yel)
M = pd.DataFrame(rows); M['aspect_diff'] = (M.aspect_marked / M.aspect_orig - 1).abs()
pd.set_option('display.width', 220); print(M.round(3).to_string())
assert M.original.is_unique, 'two marked images matched the same original - check the table'
M['name_agrees'] = [os.path.splitext(a)[0].strip() == os.path.splitext(b)[0].strip() for a, b in zip(M.marked, M.original)]
M['confirmed'] = (M.match_corr > 0.95) & (M.name_agrees | (M.match_corr - M.second_best > 0.03))
print('content match agrees with file name:', int(M.name_agrees.sum()), 'of', len(M))
assert M.confirmed.all(), 'uncertain match - check the table'
print('max aspect difference:', round(M.aspect_diff.max(), 4), '(> 0.02 means the phone cropped the image)')
M.to_csv(f'{RES}/match_table.csv', index=False)

In [ ]:
# 2 · Strokes -> regions on the full-resolution original
def regions(fname):
    gray = ORIG[fname]; H, W = gray.shape; bgr, red, yel = MK[fname]
    red = cv2.resize(red.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
    yel = cv2.resize(yel.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
    proxy, _, _ = generate_breast_mask(gray, 20)
    cw = chest_wall_side(proxy)                      # side of the image where the chest wall is
    def row_boundary(line, gap_to_edge=0.03):
        # per image row: the extreme stroke pixel away from the chest wall; rows between strokes interpolated (robust to gaps);
        # rows beyond the stroke's ends filled only when the stroke ends close to the top/bottom edge
        ys, xs = np.nonzero(line)
        if ys.size < 20: return None
        rows = np.unique(ys)
        ext = np.array([xs[ys == y].max() if cw == 'left' else xs[ys == y].min() for y in rows], float)
        y0, y1 = rows.min(), rows.max()
        if y0 <= gap_to_edge * H: y0 = 0
        if y1 >= (1 - gap_to_edge) * H - 1: y1 = H - 1
        yy = np.arange(y0, y1 + 1); xb = np.interp(yy, rows, ext)
        xb = cv2.medianBlur(xb.astype(np.float32).reshape(-1, 1), 5).ravel() if xb.size >= 5 else xb
        m = np.zeros((H, W), bool); cols = np.arange(W)[None, :]
        m[yy] = (cols <= xb[:, None]) if cw == 'left' else (cols >= xb[:, None])
        return m
    k = max(3, int(0.006 * max(H, W)) | 1); ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    zone = cv2.dilate((proxy > 0).astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(0.2 * W) | 1,) * 2)) > 0
    # zone = breast found by the rule-based mask, widened by 10% of the image width on every side
    def clean(line, red_line):
        # drop marks that are not the boundary line, above all the image number written in red in the background:
        # a red piece is kept only if most of it lies in the zone around the breast; yellow pieces are kept if they are
        # at least 15% of the largest yellow piece
        n, lab, st, _ = cv2.connectedComponentsWithStats((cv2.dilate(line.astype(np.uint8), ker) > 0).astype(np.uint8), connectivity=8)
        if n <= 1: return line > 0, 0
        tot = np.bincount(lab.ravel(), minlength=n); inz = np.bincount(lab[zone], minlength=n)
        ok = inz >= 0.5 * tot if red_line else st[:, cv2.CC_STAT_AREA] >= 0.15 * st[1:, cv2.CC_STAT_AREA].max()
        ok[0] = False; keep = np.flatnonzero(ok)
        return (line > 0) & np.isin(lab, keep), int(n - 1 - len(keep))
    def main_part(m):
        # keep only the part(s) of the region that lie mostly on the breast: a red number drawn just above the breast can
        # survive the zone filter and add a separate strip in the background; such a strip barely overlaps the breast
        n, lab = cv2.connectedComponents(m.astype(np.uint8), connectivity=8)
        if n <= 2: return m, 0
        tot = np.bincount(lab.ravel(), minlength=n); inb = np.bincount(lab[proxy > 0], minlength=n)
        ok = inb >= 0.5 * tot; ok[0] = False
        if not ok.any(): ok[np.argmax(np.r_[0, tot[1:]])] = True
        return np.isin(lab, np.flatnonzero(ok)), int(n - 1 - ok.sum())
    def flood(line):                                    # region enclosed by the line and the chest-wall side (valid only if the line is closed)
        wall = cv2.dilate(line.astype(np.uint8), ker) > 0
        n, lab = cv2.connectedComponents((~wall).astype(np.uint8), connectivity=4)
        bg = set(np.unique(lab[:, [W - 1] if cw == 'left' else [0]])) - {0}
        reg = ~np.isin(lab, list(bg)); reg &= ~(wall & (cv2.dilate(np.isin(lab, list(bg)).astype(np.uint8), ker) > 0))
        n2, lab2 = cv2.connectedComponents(reg.astype(np.uint8), connectivity=4)        # drop enclosed holes (e.g. inside digits)
        side = set(np.unique(lab2[:, [0] if cw == 'left' else [W - 1]])) - {0}
        return np.isin(lab2, list(side))
    red, n_drop_red = clean(red, True); yel, n_drop_yel = clean(yel, False) if yel.sum() > 50 else (yel > 0, 0)
    rb = row_boundary(red)
    if rb is None: rb = np.zeros((H, W), bool)
    fl = flood(red)
    # a closed line gives the exact enclosed region; if the line has a gap the fill leaks and leaves only a thin strip,
    # so the closed-line region is used only when it covers at least a quarter of the rule-based breast area
    method = 'closed line' if fl.sum() >= 0.25 * max((proxy > 0).sum(), 1) else 'row boundary'
    breast_full, n_strip = main_part(fl if method == 'closed line' else rb)
    pect = np.zeros_like(breast_full)
    if yel.sum() > 50:
        pm = row_boundary(yel, gap_to_edge=0.15)          # pectoral line runs from the top edge down to the chest-wall edge
        if pm is not None:
            ys = np.nonzero(yel)[0]; pm[ys.max() + 1:] = False
            pect = pm & breast_full
    return dict(full=breast_full, pect=pect, cw=cw, proxy=proxy > 0, gray=gray, red=red, yel=yel, method=method, frac_row=rb.mean(), frac_flood=fl.mean(), dropped=n_drop_red + n_drop_yel, strips=n_strip)
R2 = {f: regions(f) for f in sub.original_filename}
for f, r in R2.items():
    print(f'{f:28s} chest wall {r["cw"]:5s} | breast {r["full"].mean():.3f} | pectoral {r["pect"].mean():.3f} | marks removed {r["dropped"]} | strips removed {r["strips"]} | {r["method"]:12s} (row {r["frac_row"]:.3f}, closed {r["frac_flood"]:.3f}) | view {sub.set_index("original_filename").loc[f, "view"]}')

In [ ]:
# 3 · Same 256 x 256 mapping as reader1 (notebook P1-23); agreement with reader1, the rule-based proxy and every saved run
def to256(mask, side):
    return (cv2.resize(pad_to_square(mask.astype(np.uint8) * 255, side), (256, 256), interpolation=cv2.INTER_NEAREST) > 127)
def dice(a, b):
    s = a.sum() + b.sum(); return 1.0 if s == 0 else 2 * (a & b).sum() / s
def raster(j, shape):
    H, W = shape; m = {'breast': np.zeros((H, W), np.uint8), 'pectoral': np.zeros((H, W), np.uint8)}
    for s in j['shapes']:
        lab = s['label'].strip().lower()
        if lab in m and s.get('shape_type', 'polygon') == 'polygon': cv2.fillPoly(m[lab], [np.round(np.array(s['points'])).astype(np.int32)], 1)
    return m
A, rows = {}, []
for _, row in sub.iterrows():
    f = row.original_filename; r = R2[f]; side = 'left' if r['cw'] == 'right' else 'right'
    j = json.load(open(f'{OUT}/annot_test/reader1/{row.file_id}.json')) if os.path.exists(f'{OUT}/annot_test/reader1/{row.file_id}.json') else \
        next(json.load(open(p)) for p in glob.glob(f'{OUT}/annot_test/reader1/*.json') if canonical_id(os.path.basename(p)) == row.file_id)
    m1 = raster(j, r['gray'].shape); f1 = (m1['breast'] | m1['pectoral']) > 0; b1 = (m1['breast'] > 0) & ~(m1['pectoral'] > 0)
    f2 = r['full']; b2 = r['full'] & ~r['pect']
    A[('reader1', row.file_id)] = dict(full=to256(f1, side), breast=to256(b1, side))
    A[('reader2', row.file_id)] = dict(full=to256(f2, side), breast=to256(b2, side))
    saved = cv2.imread(f'{ROOT}/results/data/masks/{row.file_id}.png', 0) > 127
    rows.append(dict(file_id=row.file_id, view=row['view'], patient_id=row.patient_id,
                     inter_full_native=dice(f1, f2), inter_breast_native=dice(b1, b2),
                     inter_full=dice(A[('reader1', row.file_id)]['full'], A[('reader2', row.file_id)]['full']),
                     inter_breast=dice(A[('reader1', row.file_id)]['breast'], A[('reader2', row.file_id)]['breast']),
                     pect_r1=bool(m1['pectoral'].any()), pect_r2=bool(r['pect'].any()),
                     rule_vs_r1_full=dice(saved, A[('reader1', row.file_id)]['full']), rule_vs_r2_full=dice(saved, A[('reader2', row.file_id)]['full'])))
IR = pd.DataFrame(rows); IR.to_csv(f'{RES}/inter_reader.csv', index=False)
print(IR.round(3).to_string()); print(IR.drop(columns=['file_id', 'view', 'patient_id']).describe().round(4).to_string())
print('by view:'); print(IR.groupby('view')[['inter_full', 'inter_breast']].agg(['mean', 'min']).round(4).to_string())
S = []
for d in sorted(glob.glob(f'{OUT}/runs/*_seed*')):
    fz = f'{d}/test_prob_uint8.npz'
    if not os.path.exists(f'{d}/run.json') or not os.path.exists(fz): continue
    z = np.load(fz, allow_pickle=True); P = z['prob'] >= 128; fids = [str(x) for x in z['file_id']]
    name = os.path.basename(d); exp, mdl, s = name.split('_')
    for k, fid in enumerate(fids):
        for rd in ('reader1', 'reader2'):
            if (rd, fid) in A:
                S.append(dict(run=name, exp=exp, model=mdl, seed=int(s[4:]), reader=rd, file_id=fid,
                              dice_full=dice(P[k], A[(rd, fid)]['full']), dice_breast=dice(P[k], A[(rd, fid)]['breast'])))
S = pd.DataFrame(S); S.to_csv(f'{RES}/scores_20_both_readers.csv.gz', index=False)
print(S.groupby(['exp', 'model', 'reader'])[['dice_full', 'dice_breast']].mean().unstack('reader').round(4).to_string())

In [ ]:
# 4 · QC sheets (stay in Drive): reader1 green/orange, reader2 magenta/cyan, on the original image; check every panel by eye
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fids = list(sub.file_id); fn = dict(zip(sub.file_id, sub.original_filename))
for p in range(0, 20, 10):
    fig, ax = plt.subplots(2, 5, figsize=(20, 9))
    for a, fid in zip(ax.ravel(), fids[p:p + 10]):
        r = R2[fn[fid]]; g = r['gray']; s = 700 / max(g.shape); im = cv2.cvtColor(cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_AREA), cv2.COLOR_GRAY2RGB)
        j = next(json.load(open(q)) for q in glob.glob(f'{OUT}/annot_test/reader1/*.json') if canonical_id(os.path.basename(q)) == fid)
        m1 = raster(j, g.shape)
        for msk, col, th in (((m1['breast'] | m1['pectoral']) > 0, (60, 220, 60), 2), (m1['pectoral'] > 0, (255, 160, 0), 2), (r['full'], (255, 0, 255), 1), (r['pect'], (0, 255, 255), 1)):
            mm = cv2.resize(msk.astype(np.uint8), (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)
            cs, _ = cv2.findContours(mm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(im, cs, -1, col, th)
        a.imshow(im); a.set_title(f"{fid}  Dice full {IR.set_index('file_id').loc[fid, 'inter_full']:.3f}", fontsize=8)
    for a in ax.ravel(): a.axis('off')
    plt.tight_layout(); plt.savefig(f'{RES}/qc_reader2_{p // 10 + 1}.png', dpi=90); plt.close()
print('QC sheets:', sorted(os.listdir(RES)))
print('DONE')